# UC2 — 03 Anomaly Scoring

Combines four signals into the final anomaly score:

1. **Posterior dominance** — mean smoothed posterior mass on high-risk HMM states.
2. **Rule violations** — Pattern HIGH + MEDIUM infraction counts.
3. **Gaming ratio** — fraction of activations in the 16-30 s dodge band.
4. **Burst counts** — kept but de-weighted when a rider's score is driven purely by bursts.


## 1 · Setup

In [1]:
import sys, pickle
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd

from uc2_symbols import SYMBOLS
from uc2_scoring import (
    posterior_state_dominance,
    identify_high_risk_states,
    combined_anomaly_score,
)

OUT = PROJECT_ROOT / 'outputs'

## 2 · Load inputs

In [2]:
feature_table = pd.read_parquet(OUT / 'feature_table.parquet')
z = np.load(OUT / 'sequences.npz', allow_pickle=True)
account_ids  = z['account_ids']
concatenated = z['concatenated'].ravel()
lengths      = z['lengths']

with open(OUT / 'hmm_best.pkl', 'rb') as f:
    bundle = pickle.load(f)
model = bundle['model']

# split the concatenated observations back into per-rider sequences
sequences = []
offset = 0
for L in lengths:
    sequences.append(concatenated[offset:offset + L])
    offset += L
print(f'scoring {len(sequences):,} riders')

scoring 98,241 riders


## 3 · Identify high-risk HMM states

Symbols considered fraud-shaped: `ACTIVATE_FAST_HANDHELD`, `ACTIVATE_GAMING_THRESHOLD`, `PURCHASE_THEN_ACTIVATE_FAST`. High-risk states are the HMM states that assign the most emission mass to this set.


In [3]:
fraud_symbols = [
    SYMBOLS['ACTIVATE_FAST_HANDHELD'],
    SYMBOLS['ACTIVATE_GAMING_THRESHOLD'],
    SYMBOLS['PURCHASE_THEN_ACTIVATE_FAST'],
]
high_risk_states = identify_high_risk_states(model, fraud_symbols)
print('high-risk states:', high_risk_states)

high-risk states: [0, 3, 5, 7]


## 4 · Posterior dominance per rider

In [4]:
posterior_scores = posterior_state_dominance(model, sequences, high_risk_states)
print(f'posterior score stats: min={posterior_scores.min():.3f}, median={np.median(posterior_scores):.3f}, max={posterior_scores.max():.3f}')

posterior score stats: min=0.000, median=0.829, max=1.000


## 5 · Combined anomaly score (with burst de-weight)


In [5]:
scored = feature_table.loc[account_ids].copy()
scored['posterior_dominance'] = posterior_scores

burst_cnt   = scored.get('n_OTHER_HANDHELD_PATTERN', pd.Series(0, index=scored.index)).to_numpy()
fast_cnt    = scored.get('n_ACTIVATE_FAST_HANDHELD',    pd.Series(0, index=scored.index)).to_numpy()
gaming_cnt  = scored.get('n_ACTIVATE_GAMING_THRESHOLD', pd.Series(0, index=scored.index)).to_numpy()

scored['combined_anomaly_score'] = combined_anomaly_score(
    posterior_dominance  = posterior_scores,
    rule_violation_count = scored['max_infractions_240h'].to_numpy() + scored['max_infractions_168h'].to_numpy(),
    gaming_ratio         = scored['near_threshold_ratio'].to_numpy(),
    burst_counts         = burst_cnt,
    fast_counts          = fast_cnt,
    gaming_counts        = gaming_cnt,
)

scored.sort_values('combined_anomaly_score', ascending=False).head(20)[['n_activations', 'max_infractions_240h', 'near_threshold_ratio', 'posterior_dominance', 'combined_anomaly_score']]

,n_activations,max_infractions_240h,near_threshold_ratio,posterior_dominance,combined_anomaly_score
account_id,,,,,
72699c319ec6b808cc4ada4a62f36b585e62cfa8334d657f2965e47610eed4d1,24,5,0.260870,0.995020,0.954332
5fd6ad5ae92460dd7535b78847eb7b6546013c409e044943e74c520d736de973,19,3,0.166667,0.997723,0.952560
f6771b21d50d80d0c7ac9179f6c92b0f6ffc3de8a844ed793391d2b1279a133a,9,4,0.375000,0.996420,0.951908
23ba9ce541ac9822922c3338701182ea20c2ea1e51ba5c0109c1acdb22f57464,16,3,0.200000,0.998922,0.951563
c9568e031558c1b0b1f1f53340d0117ee1c9db035fd54b9fa84cb0c41a8aa4a3,9,5,0.125000,0.998882,0.951563
c9e66804187ed9cf4ed4562a8f0c48a3bd8f0d8a4a5f869839d5d607087fbe23,13,7,0.166667,0.999687,0.951563
bc7d3d515194124a241c307fa68eebe0fecc3f6851b519f263c280a389dbfb54,8,4,0.142857,0.998892,0.951563
a9530c9a4d4541b7e5e7e19c1078d3806d0232bc5d1addf9067103d2bb57568f,25,3,0.208333,0.998824,0.951549
6b7f635024553567686c00d413d2d03ed2bb6936b22d985bcd7a7dbc6026ebf7,8,5,0.142857,0.998496,0.951385


## 6 · Shortlist for human review

Top 100 riders by combined score, annotated with which signal pushed them up.

In [6]:
shortlist = scored.nlargest(100, 'combined_anomaly_score').copy()
shortlist['primary_signal'] = shortlist.apply(
    lambda r: ('posterior' if r['posterior_dominance'] > 0.3 else
               'rules'     if r['max_infractions_240h'] >= 3 else
               'gaming'    if r['near_threshold_ratio'] > 0.3 else
               'other'), axis=1)
shortlist['primary_signal'].value_counts()

primary_signal
posterior    100
Name: count, dtype: int64

## 7 · Validate the v1 false positive is no longer #1

In v1 a rider with 350 bursts but only 2 fast events topped the list at score 20.56. After applying burst de-weight + posterior scoring that rider should not rank #1 here.

In [7]:
burst_only = scored[(burst_cnt >= 100) & (fast_cnt + gaming_cnt < 3)]
burst_only_top = burst_only['combined_anomaly_score'].max() if not burst_only.empty else float('nan')
top_score      = scored['combined_anomaly_score'].max()
print(f'max combined score overall           : {top_score:.3f}')
print(f'max combined score among burst-only  : {burst_only_top:.3f}')
print('burst-only riders in top-100         :',
      shortlist.index.isin(burst_only.index).sum())

max combined score overall           : 0.954
max combined score among burst-only  : 0.000
burst-only riders in top-100         : 0


## 8 · Save

In [8]:
scored.to_parquet   (OUT / 'rider_scores.parquet')
shortlist.to_csv    (OUT / 'uc2_human_review_shortlist_v2.csv')
print('saved: rider_scores.parquet, uc2_human_review_shortlist_v2.csv')

saved: rider_scores.parquet, uc2_human_review_shortlist_v2.csv
